In [ ]:
import numpy as np
import keras
from keras.layers import Input,Dense
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
iris = load_iris()
X,y = iris.data,iris.target

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
model = keras.Sequential()
model.add(Input(shape=(4,)))
model.add(Dense(8,activation='relu'))
model.add(Dense(10,activation='relu'))
model.add(Dense(10,activation='relu'))
model.add(Dense(3,activation='softmax'))

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),loss='sparse_categorical_crossentropy',metrics=['accuracy'])

In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 8)              │            40 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │            90 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           110 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 273 (1.07 KB)

 Trainable params: 273 (1.07 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(X_train,y_train,validation_split=0.2,epochs=50,batch_size=16,verbose=1)

Epoch 1/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 102ms/step - accuracy: 0.3854 - loss: 1.1062 - val_accuracy: 0.6250 - val_loss: 1.0063
Epoch 2/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.3958 - loss: 1.0788 - val_accuracy: 0.6250 - val_loss: 0.9834
Epoch 3/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.4271 - loss: 1.0545 - val_accuracy: 0.6250 - val_loss: 0.9618
Epoch 4/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.4896 - loss: 1.0340 - val_accuracy: 0.7500 - val_loss: 0.9427
Epoch 5/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.5938 - loss: 1.0190 - val_accuracy: 0.7917 - val_loss: 0.9250
Epoch 6/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.6354 - loss: 1.0053 - val_accuracy: 0.7917 - val_loss: 0.9068
Epoch 7/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.6354 - loss: 0.9932 - val_accuracy: 0.7917 - val_loss: 0.8896
Epoch 8/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.6458 - loss: 0.9816 - val_accuracy: 0.7917 - val_loss: 0.8725

In [ ]:
loss,acc = model.evaluate(X_test,y_test)
print(f'Loss: {loss}, Accuracy: {acc}')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.9667 - loss: 0.2034
Loss: 0.20342479646205902, Accuracy: 0.9666666388511658


In [ ]:
model.save("iris_mlp.keras")

In [ ]:
reloaded = keras.models.load_model("iris_mlp.keras")
sample = scaler.transform([[5.1,3.5,1.4,0.2]])
probs = reloaded.predict(sample)[0]
print(f"Sample: {sample}")
print(f"Prediction: -> {iris.target_names[np.argmax(probs)]} = {reloaded.predict(sample)}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step
Sample: [[-0.86445224  0.98006827 -1.33331205 -1.31260282]]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
Prediction: -> setosa = [[0.97743213 0.01302818 0.00953963]]


In [ ]:
import gradio as gr
import numpy as np
CLASSES = list(iris.target_names)
def predict(sepal_length,sepal_width,petal_length,petal_width):
  X = np.array([[sepal_length,sepal_width,petal_length,petal_width]])
  probs = model.predict(scaler.transform(X))[0]
  return {CLASSES[i]: float(probs[i]) for i in range(3)}


In [ ]:
demo = gr.Interface(fn=predict,inputs=[gr.Number(),gr.Number(),gr.Number(),gr.Number()],outputs=gr.Label(num_top_classes=3,label="predicted species") )

In [ ]:
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://de073aeb9717b44115.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
